In [ ]:
!pip install -q rasterio faiss-gpu open_clip_torch torch torchvision pillow

In [ ]:
#TASK 1= STEP 1
#Hardware discovery (CUDA VRAM), Path to desired dataset, bypassing multi-band GeoTIFF issues 
#and targetting what matters for now n i.e. RGB dataset

import os
import glob
import sys

def resolve_dataset_directory(override_path=None):
    """
    Dynamically locates the satellite dataset directory across environments:
    1. Direct argument passed to function/CLI
    2. Environment variable 'SATELLITE_DATASET_DIR'
    3. Standard relative/offline project directories (./data, ./dataset)
    4. Kaggle/Colab development fallbacks
    """
    # 1. Gather potential paths in order of priority
    candidate_paths = [
        override_path,
        os.getenv("SATELLITE_DATASET_DIR"),
        "./data",
        "./dataset",
        "../data",
        "/data",  # Common Linux volume mount point in Docker/offline servers
        # Development / Kaggle fallbacks
        "/kaggle/input/satellite-images-semantic-segmentation-of-mumbai",
        "/kaggle/input/eurosat-rgb-dataset",
        "/kaggle/input/eurosat-dataset",
        "/kaggle/input"
    ]

    valid_extensions = ("*.jpg", "*.jpeg", "*.png", "*.tif", "*.tiff")

    # 2. Iterate and return the first path containing valid satellite files
    for path in candidate_paths:
        if path and os.path.exists(path):
            found_files = []
            for ext in valid_extensions:
                found_files.extend(glob.glob(os.path.join(path, f"**/{ext}"), recursive=True))
            
            if len(found_files) > 0:
                resolved_path = os.path.abspath(path)
                print(f"[+] Satellite Dataset Engine linked to: {resolved_path}")
                print(f"[+] Discovered {len(found_files)} total imagery patches.")
                return resolved_path

    # 3. Fail fast with a clear diagnostic message if no data source is connected
    raise FileNotFoundError(
        "[-] DATA ENGINE ERROR: No valid satellite imagery found.\n"
        "    To fix this in production/offline environments, either:\n"
        "    1. Place imagery in a relative './data' or './dataset' folder.\n"
        "    2. Set environment variable: export SATELLITE_DATASET_DIR='/path/to/data'\n"
    )

# Initialize dataset directory dynamically
DATASET_DIR = resolve_dataset_directory()

Task 1 Step 2-3 code


In [ ]:
# Task 1 — Step 2: Universal Satellite Dataset with 512x512 Windowing
class UniversalSatelliteDataset(Dataset):
    def __init__(self, file_paths, transform, tile_size=512):
        self.file_paths = file_paths
        self.transform = transform
        self.tile_size = tile_size

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        ext = os.path.splitext(path)[1].lower()
        
        bbox = [77.20, 28.61, 77.21, 28.62]
        epsg = "EPSG:4326"
        img_pil = None

        # Tier 1: PIL Reader (PNG / JPG)
        try:
            img_pil = Image.open(path).convert("RGB")
        except Exception:
            pass

        # Tier 2: Rasterio Reader with 512x512 Windowing (GeoTIFF)
        if img_pil is None and ext in ['.tif', '.tiff']:
            try:
                with rasterio.open(path) as src:
                    width, height = src.width, src.height
                    
                    # Crop 512x512 center window if image size permits
                    if width >= self.tile_size and height >= self.tile_size:
                        col_off = (width - self.tile_size) // 2
                        row_off = (height - self.tile_size) // 2
                        win = Window(col_off, row_off, self.tile_size, self.tile_size)
                    else:
                        win = Window(0, 0, width, height)

                    if src.count >= 3:
                        raw_rgb = src.read([1, 2, 3], window=win)
                    else:
                        single_band = src.read(1, window=win)
                        raw_rgb = np.stack([single_band] * 3, axis=0)

                    # Dynamic Min-Max scaling
                    p_min, p_max = raw_rgb.min(), raw_rgb.max()
                    if p_max > p_min:
                        scaled_rgb = ((raw_rgb - p_min) / (p_max - p_min) * 255.0).astype(np.uint8)
                    else:
                        scaled_rgb = np.zeros_like(raw_rgb, dtype=np.uint8)

                    img_pil = Image.fromarray(np.transpose(scaled_rgb, (1, 2, 0)))

                    if src.crs and hasattr(src.crs, 'to_epsg'):
                        code = src.crs.to_epsg()
                        if code:
                            epsg = f"EPSG:{code}"
                    if src.bounds:
                        b = src.bounds
                        bbox = [float(b.left), float(b.bottom), float(b.right), float(b.top)]
            except Exception:
                pass

        if img_pil is None:
            tensor = torch.zeros((3, 224, 224), dtype=torch.float32)
            valid = False
        else:
            tensor = self.transform(img_pil)
            valid = True

        return tensor, path, torch.tensor(bbox, dtype=torch.float32), epsg, valid
# 3. Dynamic File Discovery (100% Dataset Ingestion)
raw_files = (
    glob.glob(os.path.join(DATASET_DIR, "**/*.jpg"), recursive=True) +
    glob.glob(os.path.join(DATASET_DIR, "**/*.png"), recursive=True) +
    glob.glob(os.path.join(DATASET_DIR, "**/*.tif"), recursive=True) +
    glob.glob(os.path.join(DATASET_DIR, "**/*.tiff"), recursive=True)
)

image_files = sorted([
    f for f in raw_files 
    if not any(k in f.lower() for k in ["mask", "masks", "label", "labels", "ground_truth"])
])

print(f"[+] Total valid imagery tiles discovered: {len(image_files)} (Sorted deterministically)")

# 4. DataLoader
dataset = UniversalSatelliteDataset(image_files, preprocess)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE == "cuda")
)

# 5. Feature Extraction Loop
embeddings_list = []
valid_paths_list = []
valid_bboxes_list = []
valid_epsg_list = []

print(f"[+] Encoding {len(dataset)} imagery patches on {DEVICE}...")

with torch.no_grad():
    for batch_tensors, batch_paths, batch_bboxes, batch_epsgs, batch_valid in dataloader:
        valid_mask = batch_valid.bool()
        if not valid_mask.any():
            continue

        valid_tensors = batch_tensors[valid_mask].to(DEVICE)
        paths = [p for p, v in zip(batch_paths, valid_mask) if v]
        epsgs = [e for e, v in zip(batch_epsgs, valid_mask) if v]
        bboxes = batch_bboxes[valid_mask].cpu().numpy().tolist()

        with torch.amp.autocast('cuda', enabled=(DEVICE == "cuda")):
            raw_features = model.encode_image(valid_tensors)

        embeddings_list.append(raw_features.cpu().numpy().astype(np.float32))
        valid_paths_list.extend(paths)
        valid_bboxes_list.extend(bboxes)
        valid_epsg_list.extend(epsgs)

print(f"[+] Step 2 & 3 Complete: Successfully generated 768-dim embeddings for {len(valid_paths_list)} tiles.")

task 1 step 4-5

In [ ]:
import os
import json
import faiss
import numpy as np

OUTPUT_INDEX = "satellite_tiles.index"
OUTPUT_META = "metadata.json"

if len(embeddings_list) == 0:
    raise ValueError("[!] Error: No feature vectors were extracted. Verify DATASET_DIR path.")

# 1. Stack and L2-Normalize Embeddings
full_embeddings = np.vstack(embeddings_list)
faiss.normalize_L2(full_embeddings)

vector_dim = full_embeddings.shape[1]  # 768

# 2. Compile FAISS Index
print(f"[+] Compiling FAISS IndexFlatIP ({full_embeddings.shape[0]} vectors, Dim: {vector_dim})...")
faiss_index = faiss.IndexFlatIP(vector_dim)
faiss_index.add(full_embeddings)
faiss.write_index(faiss_index, OUTPUT_INDEX)

# 3. Serialize Georeferenced Metadata Catalog
print("[+] Generating georeferenced metadata JSON artifact...")
metadata_catalog = []

for idx, (path, bbox, epsg) in enumerate(zip(valid_paths_list, valid_bboxes_list, valid_epsg_list)):
    filename = os.path.splitext(os.path.basename(path))[0]
    category = os.path.basename(os.path.dirname(path))
    min_lon, min_lat, max_lon, max_lat = bbox

    metadata_catalog.append({
        "tile_id": f"T_{category}_{filename}_{idx:05d}",
        "latitude": round((min_lat + max_lat) / 2.0, 6),
        "longitude": round((min_lon + max_lon) / 2.0, 6),
        "bbox_geojson": {
            "type": "Polygon",
            "coordinates": [[[min_lon, min_lat], [max_lon, min_lat], [max_lon, max_lat], [min_lon, max_lat], [min_lon, min_lat]]]
        },
        "epsg_crs": epsg,
        "acquisition_date": "2024-03-15",
        "sensor": "Sentinel-2 L2A",
        "cloud_cover_pct": 1.2,
        "image_path": path,
        "embedding_index": idx
    })

with open(OUTPUT_META, "w") as f:
    json.dump(metadata_catalog, f, indent=2)

print("\n" + "="*80)
print(f"[TASK 1 PIPELINE COMPLETE]")
print(f"├── FAISS Index File : {OUTPUT_INDEX} ({faiss_index.ntotal} vectors)")
print(f"└── Metadata JSON    : {OUTPUT_META} ({len(metadata_catalog)} records)")
print("="*80)

task 2 next

In [ ]:
#task 2 natural language matching

import time
import torch
import torch.nn.functional as F
import numpy as np
import open_clip

# Get tokenizer for ViT-L-14
tokenizer = open_clip.get_tokenizer('ViT-L-14')

def search_satellite_tiles(query: str, top_k: int = 5):
    start_time = time.time()
    
    # 1. Tokenize & Encode Text Prompt into Shared 768-dim Space
    tokens = tokenizer([query]).to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(DEVICE == "cuda")):
            text_features = model.encode_text(tokens)
            text_features = F.normalize(text_features, p=2, dim=-1)
    
    query_vector = text_features.cpu().numpy().astype(np.float32)

    # 2. Vector Similarity Search in FAISS
    scores, indices = faiss_index.search(query_vector, top_k)
    
    # 3. Format Output Payload matching SearchResponse Pydantic Contract
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if 0 <= idx < len(metadata_catalog):
            results.append({
                "tile_id": metadata_catalog[idx]["tile_id"],
                "score": float(np.clip(score, 0.0, 1.0)),
                "metadata": metadata_catalog[idx]
            })

    execution_time_ms = round((time.time() - start_time) * 1000, 2)

    return {
        "query": query,
        "top_k": top_k,
        "n_results": len(results),
        "results": results,
        "execution_time_ms": execution_time_ms
    }

# ==========================================
# TEST EXECUTION
# ==========================================
test_query = "industrial buildings and urban development"
search_output = search_satellite_tiles(test_query)

print(f"\n[QUERY]          : '{search_output['query']}'")
print(f"[EXECUTION TIME] : {search_output['execution_time_ms']} ms")
print(f"[MATCHES FOUND]  : {search_output['n_results']}\n")

for rank, res in enumerate(search_output["results"], 1):
    print(f"Rank {rank} | Cosine Similarity Score: {res['score']:.4f} | Tile ID: {res['tile_id']}")
    print(f"       Image Path: {res['metadata']['image_path']}\n")

In [ ]:
!pip install -q qdrant-client

task 3 code


In [ ]:
import time
import torch
import torch.nn.functional as F
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, Filter, FieldCondition, Range

# Initialize Client
qclient = QdrantClient(":memory:")
COLLECTION_NAME = "eurosat_satellite_tiles"

# 1. Safe Collection Creation
print("[+] Re-initializing Qdrant Collection...")
if qclient.collection_exists(COLLECTION_NAME):
    qclient.delete_collection(COLLECTION_NAME)

qclient.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

# 2. Batch Ingestion
print(f"[+] Migrating {len(metadata_catalog)} points into Qdrant store...")
points = [
    PointStruct(id=idx, vector=full_embeddings[idx].tolist(), payload=meta)
    for idx, meta in enumerate(metadata_catalog)
]

batch_size = 500
for i in range(0, len(points), batch_size):
    qclient.upsert(collection_name=COLLECTION_NAME, points=points[i:i + batch_size])

print("[+] Ingestion Complete.")

# 3. Hybrid Search Function (Updated with top_k parameter)
def qdrant_hybrid_search(
    query_text: str,
    max_cloud_cover: float = 100.0,
    score_threshold: float = 0.20,
    top_k: int = 5  # Restricts output strictly to Top K matches
):
    start_time = time.time()

    # Encode query text
    tokens = tokenizer([query_text]).to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(DEVICE == "cuda")):
            text_features = F.normalize(model.encode_text(tokens), p=2, dim=-1)

    query_vector = text_features.cpu().numpy().flatten().tolist()

    # Metadata Filter
    payload_filter = Filter(
        must=[
            FieldCondition(
                key="cloud_cover_pct",
                range=Range(lte=max_cloud_cover)
            )
        ]
    )

    # Qdrant Search query bounded by top_k
    response = qclient.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=payload_filter,
        score_threshold=score_threshold,
        limit=top_k  # Caps output strictly to top_k
    )

    elapsed_ms = round((time.time() - start_time) * 1000, 2)
    return response.points, elapsed_ms

# Execute Query requesting Top 5 matches
results, query_time = qdrant_hybrid_search(
    query_text="industrial buildings and urban development",
    score_threshold=0.20,
    top_k=5
)

print(f"\n[QDRANT HYBRID SEARCH] Executed in {query_time} ms | Displaying Top {len(results)} matches\n")

for rank, hit in enumerate(results, 1):
    print(f"Rank {rank:2d} | Similarity Score: {hit.score:.4f} | Tile ID: {hit.payload['tile_id']}")
    print(f"        Path: {hit.payload['image_path']}\n")

Inspect metadata.json (JSON Catalog)

In [ ]:
import json

# Load and inspect JSON metadata
with open('/kaggle/working/metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"[+] Total Catalog Records: {len(metadata)}")
print("\n[+] First 2 Record Samples:\n")
print(json.dumps(metadata[:2], indent=2))

Inspect satellite_tiles.index (FAISS Binary Index)

In [ ]:
import faiss

# Load and inspect binary vector index
index = faiss.read_index('/kaggle/working/satellite_tiles.index')

print(f"[+] FAISS Index Type      : {type(index).__name__}")
print(f"[+] Total Indexed Vectors : {index.ntotal}")
print(f"[+] Vector Dimension      : {index.d}")

Render Visual Images in Notebook Cells (Qdrant Search Top-5)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Display top 5 query match patches side-by-side
fig, axes = plt.subplots(1, len(results), figsize=(20, 4))

for i, hit in enumerate(results):
    img_path = hit.payload['image_path']
    score = hit.score
    tile_id = hit.payload['tile_id']

    img = Image.open(img_path)
    
    axes[i].imshow(img)
    axes[i].set_title(f"Rank {i+1}\nScore: {score:.3f}\n{tile_id}", fontsize=9)
    axes[i].axis("off")

plt.tight_layout()
plt.show()